# Convolutional Neural Networks from Scratch

Implementing CNN operations using only NumPy to understand how convolutions, pooling, and feature detection work.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from scipy.signal import correlate2d
np.random.seed(42)
%matplotlib inline

## 1. Convolution Operation from Scratch

In [ ]:
def convolve2d(image, kernel, stride=1, padding=0):
    """Implement 2D convolution (actually cross-correlation) from scratch."""
    # Add padding
    if padding > 0:
        image = np.pad(image, padding, mode='constant', constant_values=0)
    
    h, w = image.shape
    kh, kw = kernel.shape
    
    # Output dimensions
    out_h = (h - kh) // stride + 1
    out_w = (w - kw) // stride + 1
    output = np.zeros((out_h, out_w))
    
    for i in range(out_h):
        for j in range(out_w):
            region = image[i*stride:i*stride+kh, j*stride:j*stride+kw]
            output[i, j] = np.sum(region * kernel)
    
    return output

# Test with a simple example
image = np.array([[1, 2, 3, 0, 1],
                  [0, 1, 2, 3, 1],
                  [1, 0, 1, 2, 0],
                  [2, 1, 0, 1, 1],
                  [1, 1, 2, 0, 2]], dtype=float)

kernel = np.array([[1, 0, -1],
                   [1, 0, -1],
                   [1, 0, -1]], dtype=float)

result = convolve2d(image, kernel)
print(f'Input shape: {image.shape}')
print(f'Kernel shape: {kernel.shape}')
print(f'Output shape: {result.shape}')
print(f'Output:\n{result}')

## 2. Visualize Convolution with Different Kernels

In [ ]:
# Create a sample image (using digits dataset)
digits = load_digits()
sample_image = digits.images[0]  # 8x8 digit image

# Upscale for better visualization
from scipy.ndimage import zoom
sample_large = zoom(sample_image, 4, order=1)  # 32x32

# Define kernels
kernels = {
    'Edge Horizontal': np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=float),
    'Edge Vertical': np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=float),
    'Sharpen': np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=float),
    'Blur (Box)': np.ones((3, 3)) / 9,
    'Emboss': np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], dtype=float),
    'Sobel X': np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float),
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes[0, 0].imshow(sample_large, cmap='gray')
axes[0, 0].set_title('Original')
axes[1, 0].axis('off')

for idx, (name, kernel) in enumerate(kernels.items()):
    row, col = divmod(idx + 1, 4) if idx + 1 >= 4 else (0, idx + 1)
    if idx + 1 >= 4:
        row, col = 1, idx + 1 - 4 + 1
    else:
        row, col = 0, idx + 1
    output = convolve2d(sample_large, kernel, padding=1)
    axes.ravel()[idx + 1].imshow(output, cmap='gray')
    axes.ravel()[idx + 1].set_title(name)

for ax in axes.ravel():
    ax.axis('off')
plt.suptitle('Convolution with Different Kernels', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Max Pooling from Scratch

In [ ]:
def max_pool2d(image, pool_size=2, stride=2):
    """Implement max pooling from scratch."""
    h, w = image.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))
    
    for i in range(out_h):
        for j in range(out_w):
            region = image[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.max(region)
    
    return output

def avg_pool2d(image, pool_size=2, stride=2):
    """Implement average pooling from scratch."""
    h, w = image.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))
    
    for i in range(out_h):
        for j in range(out_w):
            region = image[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.mean(region)
    
    return output

# Demonstrate
test_img = np.random.rand(8, 8)
max_pooled = max_pool2d(test_img, pool_size=2, stride=2)
avg_pooled = avg_pool2d(test_img, pool_size=2, stride=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(test_img, cmap='viridis'); axes[0].set_title(f'Original {test_img.shape}')
axes[1].imshow(max_pooled, cmap='viridis'); axes[1].set_title(f'Max Pool {max_pooled.shape}')
axes[2].imshow(avg_pooled, cmap='viridis'); axes[2].set_title(f'Avg Pool {avg_pooled.shape}')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Pooling reduces spatial dimensions: {test_img.shape} -> {max_pooled.shape}')

## 4. Simple CNN Forward Pass

In [ ]:
class SimpleCNN:
    """Simple CNN: Conv -> ReLU -> Pool -> Flatten -> Dense"""
    
    def __init__(self):
        # Initialize random filters
        self.conv_filters = np.random.randn(4, 3, 3) * 0.5  # 4 filters of 3x3
        self.dense_weights = None  # initialized on first forward pass
        self.dense_bias = np.zeros(10)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def forward(self, image):
        """Forward pass through the network."""
        self.input = image
        
        # Conv layer: apply each filter
        h, w = image.shape
        self.conv_output = np.zeros((4, h-2, w-2))  # 4 feature maps
        for f in range(4):
            self.conv_output[f] = convolve2d(image, self.conv_filters[f])
        
        # ReLU
        self.relu_output = self.relu(self.conv_output)
        
        # Max Pool (2x2)
        self.pool_output = np.zeros((4, (h-2)//2, (w-2)//2))
        for f in range(4):
            self.pool_output[f] = max_pool2d(self.relu_output[f])
        
        # Flatten
        self.flat = self.pool_output.flatten()
        
        # Dense layer
        if self.dense_weights is None:
            self.dense_weights = np.random.randn(10, len(self.flat)) * 0.01
        
        # Output (logits)
        logits = self.dense_weights @ self.flat + self.dense_bias
        
        # Softmax
        exp_logits = np.exp(logits - logits.max())
        self.output = exp_logits / exp_logits.sum()
        
        return self.output

# Run forward pass on a digit
cnn = SimpleCNN()
digit_image = digits.images[0]  # 8x8
output = cnn.forward(digit_image)
print(f'Input shape: {digit_image.shape}')
print(f'Conv output: {cnn.conv_output.shape} (4 feature maps)')
print(f'After ReLU: {cnn.relu_output.shape}')
print(f'After Pool: {cnn.pool_output.shape}')
print(f'Flattened: {cnn.flat.shape}')
print(f'Output (probabilities): {output.shape}')
print(f'Predicted class: {np.argmax(output)} (actual: {digits.target[0]})')

## 5. Feature Map Visualization

In [ ]:
# Visualize what each layer sees
fig, axes = plt.subplots(4, 4, figsize=(14, 14))

# Original
axes[0, 0].imshow(digit_image, cmap='gray')
axes[0, 0].set_title('Input')
for j in range(1, 4): axes[0, j].axis('off')

# Conv outputs
for f in range(4):
    axes[1, f].imshow(cnn.conv_output[f], cmap='RdBu_r')
    axes[1, f].set_title(f'Conv Filter {f}')

# After ReLU
for f in range(4):
    axes[2, f].imshow(cnn.relu_output[f], cmap='hot')
    axes[2, f].set_title(f'After ReLU {f}')

# After pooling
for f in range(4):
    axes[3, f].imshow(cnn.pool_output[f], cmap='hot')
    axes[3, f].set_title(f'After Pool {f}')

for ax in axes.ravel(): ax.axis('off')
plt.suptitle('Feature Maps at Each Layer', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. How Filters Detect Features

In [ ]:
# Show how specific kernels detect edges, corners, etc.
# Create synthetic patterns
patterns = {
    'Vertical Edge': np.zeros((16, 16)),
    'Horizontal Edge': np.zeros((16, 16)),
    'Corner': np.zeros((16, 16)),
    'Diagonal': np.zeros((16, 16)),
}
patterns['Vertical Edge'][:, 8:] = 1
patterns['Horizontal Edge'][8:, :] = 1
patterns['Corner'][8:, 8:] = 1
patterns['Diagonal'] = np.fromfunction(lambda i, j: (i < j).astype(float), (16, 16))

# Apply edge detection kernels
edge_kernels = {
    'Vertical Detector': np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=float),
    'Horizontal Detector': np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=float),
}

fig, axes = plt.subplots(len(patterns), 3, figsize=(12, 14))
for i, (pname, pattern) in enumerate(patterns.items()):
    axes[i, 0].imshow(pattern, cmap='gray')
    axes[i, 0].set_title(f'Pattern: {pname}')
    for j, (kname, kernel) in enumerate(edge_kernels.items()):
        response = convolve2d(pattern, kernel, padding=1)
        axes[i, j+1].imshow(response, cmap='RdBu_r')
        axes[i, j+1].set_title(f'{kname}')

for ax in axes.ravel(): ax.axis('off')
plt.suptitle('Filter Responses to Different Patterns', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Stride and Padding Effects

In [ ]:
# Visualize effect of stride and padding
test_image = digits.images[3]  # 8x8 image
kernel_3x3 = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=float)  # Sobel

configs = [
    ('No padding, stride=1', 0, 1),
    ('Padding=1, stride=1', 1, 1),
    ('No padding, stride=2', 0, 2),
    ('Padding=1, stride=2', 1, 2),
]

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
axes[0].imshow(test_image, cmap='gray')
axes[0].set_title(f'Original {test_image.shape}')

for idx, (title, pad, stride) in enumerate(configs):
    result = convolve2d(test_image, kernel_3x3, stride=stride, padding=pad)
    axes[idx+1].imshow(result, cmap='RdBu_r')
    axes[idx+1].set_title(f'{title}\nOutput: {result.shape}')

for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

# Formula
print('Output size formula: ((W - K + 2P) / S) + 1')
print('W=input, K=kernel, P=padding, S=stride')
for title, p, s in configs:
    out = ((8 - 3 + 2*p) // s) + 1
    print(f'  {title}: ((8-3+{2*p})/{s})+1 = {out}')

## 8. Parameter Counting for CNN Architectures

In [ ]:
def count_cnn_params(architecture):
    """Count parameters in a CNN architecture."""
    total_params = 0
    print(f'{"Layer":<25} {"Output Shape":<20} {"Parameters":<15}')
    print('-' * 60)
    
    for layer in architecture:
        name = layer['name']
        params = layer['params']
        shape = layer['output_shape']
        total_params += params
        print(f'{name:<25} {str(shape):<20} {params:<15,}')
    
    print('-' * 60)
    print(f'{"TOTAL":<25} {"":<20} {total_params:<15,}')
    return total_params

# LeNet-5 like architecture
print('=== LeNet-5 Architecture ===')
lenet = [
    {'name': 'Conv2D(1,6,5x5)', 'output_shape': (6, 28, 28), 'params': 6*(1*5*5+1)},
    {'name': 'MaxPool(2x2)', 'output_shape': (6, 14, 14), 'params': 0},
    {'name': 'Conv2D(6,16,5x5)', 'output_shape': (16, 10, 10), 'params': 16*(6*5*5+1)},
    {'name': 'MaxPool(2x2)', 'output_shape': (16, 5, 5), 'params': 0},
    {'name': 'Flatten', 'output_shape': (400,), 'params': 0},
    {'name': 'Dense(400,120)', 'output_shape': (120,), 'params': 400*120+120},
    {'name': 'Dense(120,84)', 'output_shape': (84,), 'params': 120*84+84},
    {'name': 'Dense(84,10)', 'output_shape': (10,), 'params': 84*10+10},
]
count_cnn_params(lenet)

print('\n\n=== VGG-like (simplified) ===')
vgg_simple = [
    {'name': 'Conv2D(3,64,3x3)', 'output_shape': (64, 224, 224), 'params': 64*(3*3*3+1)},
    {'name': 'Conv2D(64,64,3x3)', 'output_shape': (64, 224, 224), 'params': 64*(64*3*3+1)},
    {'name': 'MaxPool(2x2)', 'output_shape': (64, 112, 112), 'params': 0},
    {'name': 'Conv2D(64,128,3x3)', 'output_shape': (128, 112, 112), 'params': 128*(64*3*3+1)},
    {'name': 'Conv2D(128,128,3x3)', 'output_shape': (128, 112, 112), 'params': 128*(128*3*3+1)},
    {'name': 'MaxPool(2x2)', 'output_shape': (128, 56, 56), 'params': 0},
    {'name': 'Flatten', 'output_shape': (128*56*56,), 'params': 0},
    {'name': 'Dense(401408,4096)', 'output_shape': (4096,), 'params': 128*56*56*4096+4096},
    {'name': 'Dense(4096,10)', 'output_shape': (10,), 'params': 4096*10+10},
]
count_cnn_params(vgg_simple)

In [ ]:
# Visualize where parameters live in a CNN
layers = ['Conv1\n(6 filters)', 'Conv2\n(16 filters)', 'FC1\n(400->120)', 'FC2\n(120->84)', 'FC3\n(84->10)']
params = [156, 2416, 48120, 10164, 850]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(layers, params, color=['steelblue', 'steelblue', 'coral', 'coral', 'coral'])
ax.set_ylabel('Number of Parameters')
ax.set_title('LeNet-5: Parameter Distribution (FC layers dominate!)')
for bar, p in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500, 
            f'{p:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()
print(f'Conv layers: {sum(params[:2]):,} params ({sum(params[:2])/sum(params)*100:.1f}%)')
print(f'FC layers: {sum(params[2:]):,} params ({sum(params[2:])/sum(params)*100:.1f}%)')

## Summary

**Key CNN Concepts:**
- Convolution extracts local features using learnable filters
- Pooling reduces spatial dimensions and provides translation invariance
- Deeper layers detect increasingly complex patterns
- Stride and padding control output dimensions
- Most parameters typically live in fully-connected layers

**Output size formula:** `((W - K + 2P) / S) + 1`